# Assignment 3: Tiny Transformer — Tiny Shakespeare
Next-token prediction with a hand-built Transformer in PyTorch.

## Phase 1 — Setup & Reproducibility

In [1]:
import math
import random
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


In [2]:
# Central config — all hyperparameters live here
config = {
    "vocab_size":  500,
    "seq_len":      50,
    "d_model":     128,
    "n_heads":       4,
    "n_layers":      2,
    "ffn_dim":     256,   # 2 × d_model
    "dropout":     0.1,
    "lr":         3e-4,
    "weight_decay": 0.01,
    "batch_size":   64,
    "num_epochs":   20,
}

## Phase 2 — Data Preparation & Tokenization

### 2a. Load the Tiny Shakespeare corpus

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)
corpus_path = DATA_DIR / "input.txt"

if not corpus_path.exists():
    URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(URL, corpus_path)

with open(corpus_path) as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")
print("Sample:")
print(text[:300])

Corpus length: 1,115,394 characters
Sample:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


### 2b. BPE Tokenization

In [4]:
tokenizer_path = DATA_DIR / "tokenizer.json"

if not tokenizer_path.exists():
    print("Training BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=config["vocab_size"],
        special_tokens=["[UNK]", "[PAD]"],
    )
    tokenizer.train_from_iterator([text], trainer=trainer)
    tokenizer.save(str(tokenizer_path))
    print(f"Tokenizer saved to {tokenizer_path}")
else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    print(f"Tokenizer loaded from {tokenizer_path}")

actual_vocab_size = tokenizer.get_vocab_size()
print(f"Vocab size: {actual_vocab_size}")
# Update config in case the trainer produced fewer tokens
config["vocab_size"] = actual_vocab_size

Tokenizer loaded from ../data/tokenizer.json
Vocab size: 500


### 2c. Encode corpus and create sliding-window sequences

In [5]:
encoding = tokenizer.encode(text)
token_ids = encoding.ids
print(f"Total tokens: {len(token_ids):,}")

SEQ_LEN = config["seq_len"]

# Build (input, target) pairs with stride 1
inputs  = []
targets = []
for i in range(len(token_ids) - SEQ_LEN):
    inputs.append(token_ids[i : i + SEQ_LEN])
    targets.append(token_ids[i + 1 : i + SEQ_LEN + 1])

inputs  = torch.tensor(inputs,  dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)
print(f"inputs shape:  {inputs.shape}")
print(f"targets shape: {targets.shape}")

Total tokens: 447,717


inputs shape:  torch.Size([447667, 50])
targets shape: torch.Size([447667, 50])


### 2d. Train / Val split (80 / 20)

In [6]:
n = len(inputs)
split = int(0.8 * n)

train_inputs,  val_inputs  = inputs[:split],  inputs[split:]
train_targets, val_targets = targets[:split], targets[split:]

print(f"Train sequences: {len(train_inputs):,}")
print(f"Val   sequences: {len(val_inputs):,}")

Train sequences: 358,133
Val   sequences: 89,534


In [7]:
class ShakespeareDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs  = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


BATCH = config["batch_size"]

train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset   = ShakespeareDataset(val_inputs,   val_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 5595
Val   batches: 1399


### 2e. Checkpoint — verify batch shapes

In [8]:
xb, yb = next(iter(train_loader))
print(f"Input  batch shape: {xb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")
print(f"Target batch shape: {yb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")

# Decode the first sequence as a sanity check
sample_tokens = xb[0].tolist()
decoded = tokenizer.decode(sample_tokens)
print(f"\nDecoded sample input:\n{decoded}")

Input  batch shape: torch.Size([64, 50])   (expected: [64, 50])
Target batch shape: torch.Size([64, 50])   (expected: [64, 50])

Decoded sample input:
an un li ck ' d be ar - w he l p That c ar ri es no im pr ess ion like the d am . And am I then a man to be be love d ? O m on str ous fa ul t , to
